https://madys.readthedocs.io/en/latest/computation_run.html

* Use madys to confirm the youth using stellar evolutionary model plotted on CMD of <0.1Gyr UBC clusters
* Use other cluster catalogs too

In [ ]:
import madys
getattr(madys, 'MADYS_VERSION', getattr(madys, '__version__', None))

In [ ]:
import sys

basedir = '/ut2/jerome/github/research/project/wakai'
sys.path.insert(0, f'{basedir}/code/')

In [ ]:
!python --version

In [ ]:
gaiaDR3id = 3762515188088861184

## All members

In [ ]:
star_list = d.Gaia.apply(lambda x: f"Gaia DR3 {x}").tolist()
star_list[:5]

In [ ]:
# because the host is not a cluster member
star_list.append(f'Gaia DR3 {gaiaDR3id}')

In [ ]:
p = madys.SampleObject(star_list, id_type='DR3', ext_map='leike')
p.phot_table

In [ ]:
p.phot_table.to_pandas().shape

In [ ]:
p.abs_phot.shape

In [ ]:
p.abs_phot_err.shape

In [ ]:
import numpy as np

abs_phot = np.c_[p.abs_phot, p.abs_phot_err]
columns = np.c_[p.filters,[i+'_err' for i in p.filters]].T.flatten()
phot = pd.DataFrame(abs_phot, columns=columns)
# phot.to_csv(f'../../data/abs_phot_{cname}_members_from_perren2023.csv', index=False)

In [ ]:
target = pd.DataFrame(p.abs_phot, columns=p.filters).squeeze()
target

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import mark_inset

# model = 'pm13'
# model = 'dartmouth'
# model = 'bt-settl'
# model = 'ames-cond'
# model = 'ames-dusty'
# model = 'nextgen'
model = 'parsec2'
x, y = 'Gbp-Grp', 'G'
plot_ages=[10, 20, 30, 100]
plot_masses=[0.3, 0.5, 0.7, 0.8, 1.0, 1.5, 2]

fig, ax = p2.CMD(
    x, y,
    model,
    xlim=[-0.3, 4.1],
    ylim=[14, -2],
    close=False,
    plot_ages=plot_ages,
    plot_masses=plot_masses,
    label_points=False,
    alpha=0.5,
    s=8,
    figsize=(8, 8)
)

xx = target.Gbp - target.Grp
yy = target.G

ax.plot(
    xx,
    yy,
    marker=r"$\star$",
    mec='k',
    c='r',
    ms=25,
    label='TOI-6557',
    zorder=10,
)

# =========================
# INSET
# =========================

# 1. Define zoom size
dx, dy = 0.4, 0.5

zoom_xlim = [xx - dx, xx + dx + 0.1]
zoom_ylim = [yy + dy, yy - dy]  # magnitude axis (bright at top)

# 2. Create inset axis
axins = ax.inset_axes([0.6, 0.6, 0.35, 0.35])

# 3. Plot content inside inset
madys.IsochroneGrid.plot_isochrones(x, y, 
                                    model, 
                                    plot_ages=plot_ages,
                                    plot_masses=plot_masses,
                                    ax=axins)
madys.SampleObject.plot_photometry(
    x, y, axins, p2,
    errors=None,
    ids=None,
    label_points=False
)

axins.plot(
    xx,
    yy,
    marker=r"$\star$",
    mec='k',
    c='r',
    ms=35,
    zorder=10,
)

# 4. Apply limits (do NOT invert again)
axins.set_xlim(zoom_xlim)
axins.set_ylim(zoom_ylim)

axins.set_facecolor('white')

# 5. Draw connector
# ax.indicate_inset_zoom(axins, edgecolor="black")

# mark_inset(
#     ax, axins,
#     loc1=4, loc2=4,  # try combinations if needed
#     fc="none",
#     ec="black",
#     lw=1
# )

ax.legend(loc='lower left')
ax.set_ylabel(r"$G$")
ax.set_xlabel(r"$G_{BP} - G_{RP}$")
ax.set_title(f"{cname} | {model.upper()} isochrones")
# fig.savefig(f"../../figures/cmd_{model}_{cname}_members_from_perren2023_inset.png", dpi=200)